In [2]:
import time
import requests
import pandas as pd
import yfinance as yf

In [3]:
# 1) Downaload wikipedia table with a User-Agent
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
resp.raise_for_status()

# 2) Get symbols from the table
sp500 = pd.read_html(resp.text, match="Symbol")[0]
sp500_tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()

# 3) Download fundamental data using yfinance
fundamentals = {}
for t in sp500_tickers:
    try:
        info = yf.Ticker(t).get_info()
        fundamentals[t] = {
            "Market Cap": info.get("marketCap"),
            "P/E Ratio": info.get("trailingPE"),
            "EPS": info.get("trailingEps"),
            "Dividend Yield": info.get("dividendYield"),
            "ROE": info.get("returnOnEquity"),
            "Price": info.get("currentPrice"),
            "Revenue": info.get("totalRevenue"),
            "Net income": info.get("netIncomeToCommon"),
            "Total Assets": info.get("totalAssets"),
            "Total Liabilities": info.get("totalLiab"),
            "Shareholders Equity": info.get("totalStockholderEquity"),
            "Cash Flow": info.get("operatingCashflow"),
            "Sector": sp500.loc[sp500["Symbol"] == t.replace("-", "."), "GICS Sector"].values[0],
            "Sub Industry": sp500.loc[sp500["Symbol"] == t.replace("-", "."), "GICS Sub-Industry"].values[0]
        }
        time.sleep(0.2)  
    except Exception as e:
        print(f"Error fetching data for {t}: {e}")

fundamentals_df = pd.DataFrame.from_dict(fundamentals, orient="index")
fundamentals_df

/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_23074/3899981035.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(resp.text, match="Symbol")[0]


,Market Cap,P/E Ratio,EPS,Dividend Yield,ROE,Price,Revenue,Net income,Total Assets,Total Liabilities,Shareholders Equity,Cash Flow,Sector,Sub Industry
MMM,82839945216,21.601389,7.20,1.88,0.94760,155.53,24601999360,3939000064,None,None,None,-1002000000,Industrials,Industrial Conglomerates
AOS,9989867520,19.857939,3.59,1.91,0.27603,71.29,3790200064,518600000,None,None,None,596099968,Industrials,Building Products
ABT,230889439232,16.644918,7.97,1.78,0.30931,132.66,43108999168,13927999488,None,None,None,9036999680,Health Care,Health Care Equipment
ABBV,371684212736,100.190475,2.10,3.12,1.12854,210.40,58327998464,3723000064,None,None,None,19282999296,Health Care,Biotechnology
ACN,161923088384,20.665342,12.58,2.28,0.26928,259.97,68482535424,7948770816,None,None,None,10949683200,Information Technology,IT Consulting & Other Services
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XYL,34457120768,36.864582,3.84,1.13,0.08625,141.56,8730000384,938000000,None,None,None,1224000000,Industrials,Industrial Machinery & Supplies & Components
YUM,40789467136,28.988165,5.07,1.93,NaN,146.97,7907999744,1432999936,None,None,None,1834000000,Consumer Discretionary,Restaurants
ZBRA,16122504192,29.942398,10.59,NaN,0.15875,317.09,5190000128,548000000,None,None,None,925000000,Information Technology,Electronic Equipment & Instruments
ZBH,21017985024,25.878050,4.10,0.90,0.06529,106.10,7833800192,823500032,None,None,None,1663000064,Health Care,Health Care Equipment


In [12]:
fundamentals_df.index.name = "Ticker"
fundamentals_df.reset_index(inplace=True)
fundamentals_df

ValueError: cannot insert Ticker, already exists

In [13]:
fundamentals_df

,Ticker,Market Cap,P/E Ratio,EPS,Dividend Yield,ROE,Price,Revenue,Net income,Total Assets,Total Liabilities,Shareholders Equity,Cash Flow,Sector,Sub Industry
Ticker,,,,,,,,,,,,,,,
0,MMM,82839945216,21.601389,7.20,1.88,0.94760,155.53,24601999360,3939000064,None,None,None,-1002000000,Industrials,Industrial Conglomerates
1,AOS,9989867520,19.857939,3.59,1.91,0.27603,71.29,3790200064,518600000,None,None,None,596099968,Industrials,Building Products
2,ABT,230889439232,16.644918,7.97,1.78,0.30931,132.66,43108999168,13927999488,None,None,None,9036999680,Health Care,Health Care Equipment
3,ABBV,371684212736,100.190475,2.10,3.12,1.12854,210.40,58327998464,3723000064,None,None,None,19282999296,Health Care,Biotechnology
4,ACN,161923088384,20.665342,12.58,2.28,0.26928,259.97,68482535424,7948770816,None,None,None,10949683200,Information Technology,IT Consulting & Other Services
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYL,34457120768,36.864582,3.84,1.13,0.08625,141.56,8730000384,938000000,None,None,None,1224000000,Industrials,Industrial Machinery & Supplies & Components
499,YUM,40789467136,28.988165,5.07,1.93,NaN,146.97,7907999744,1432999936,None,None,None,1834000000,Consumer Discretionary,Restaurants
500,ZBRA,16122504192,29.942398,10.59,NaN,0.15875,317.09,5190000128,548000000,None,None,None,925000000,Information Technology,Electronic Equipment & Instruments


In [16]:
tickers = fundamentals_df[["Ticker","Sector","Sub Industry"]]
tickers = pd.DataFrame(tickers)
tickers.to_csv("sp500_tickers.csv", index=False)